<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/pset2/2026_pset2_empty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Set 2

52 Points (4 problems × 13 points each)

**Due**: 2026.03.16 at 11:59pm

**Collaboration**: Collaboration is welcome. While we encourage you to work together, share ideas, and learn from each other, all short answers and code must be written individually. As is good practice in research, we expect you to acknowledge your collaborators in the cell below.

**Grading:** Each problem is worth 13 points. You may **skip one problem** of your choice — your grade will be based on your best 3 out of 4 (39 points max). If you complete all 4, we will drop the lowest.

**Instructions:**
1. Make a copy of this notebook in your Google Drive
2. Run the first code cell to download the data and import packages
3. Complete each <font color="red">TODO</font> section — replace `...` with your code or write your answer
4. Download as PDF and submit to Gradescope

- File > Download > Download .ipynb
- Convert .ipynb file to PDF with [this converter](https://ovvens.com/colab-converter/)
- Upload the PDF to Gradescope

**Collaborators:**

<font color="red">
    TODO
</font>

In [ ]:
# Download data files
!wget -qnc https://raw.githubusercontent.com/sokrypton/7.571/refs/heads/main/pset2/Q1_MAPs.csv.gz
!wget -qnc https://raw.githubusercontent.com/sokrypton/7.571/refs/heads/main/pset2/Q1_mESC_raw_data.csv.gz
!wget -qnc https://raw.githubusercontent.com/sokrypton/7.571/refs/heads/main/pset2/Q3.csv.gz

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
import scipy.stats as st
from IPython.display import Image

plt.rcParams['figure.figsize'] = (6, 6)
plt.rcParams['font.size'] = 12

## Quick guide: Pandas DataFrames

In Problem Set 1, we loaded CSV files manually with Python's `csv` module and stored data in dictionaries of NumPy arrays. In this pset, we'll use **pandas**, a library that makes it easier to work with tabular data that has labeled rows and columns.

A pandas **DataFrame** is like a spreadsheet: it has rows, columns, and labels for both. Here are the key operations you'll need:

**Loading data:**
```python
df = pd.read_csv('file.csv', index_col=0)  # first column becomes row labels
```

**Inspecting data:**
```python
df.shape        # (num_rows, num_columns)
df.head()       # show the first 5 rows
df.index        # the row labels (e.g. gene names)
df.columns      # the column labels (e.g. sample names)
```

**Accessing data:**
```python
df.loc['Trim28']              # get one row by its label → returns a Series
df.loc['Trim28'][1]           # get a single value (Trim28 expression in column 1)
df[['glu1', 'glu2']]          # select specific columns by name
                              # Equivalently: cols = ['glu1', 'glu2']; df[cols]
```

**Computing:**
```python
df.loc['Trim28'].mean()              # mean of a row
df.loc['Trim28'].var()               # variance of a row
df[['glu1', 'glu2']].mean(axis=1)    # mean across selected columns, per row
df['new_col'] = some_array           # add a new column
```

**Filtering & sorting:**
```python
df[df['pvals'] < 0.01]                               # keep rows where condition is True
df.sort_values('column_name')                        # sort rows by a column
df.sort_values('column_name', ascending=False)[:20]  # top 20
```

We'll also use **seaborn** (`sns`), a plotting library that works well with pandas and makes it easy to create histograms (`sns.histplot`), scatter plots (`sns.scatterplot`), and clustered heatmaps (`sns.clustermap`).

# Problem 1: Bayesian Inference (13 points)

### Background

Single-cell RNA-seq (scRNA-seq) data often suffers from many missing values due to low capture efficiency (the percent of RNAs in the cell that are successfully captured and sequenced). Further, the RNA capture efficiency often varies across cells and experiments. Therefore, downstream analysis requires careful normalization, and [imputation](https://en.wikipedia.org/wiki/Imputation_(statistics)) can be used to predict missing values.

One approach to normalization and imputation, applied in [this paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7703772/), uses Bayes' theorem in their **bayNorm** package.

The idea: we can use our measured RNA counts for a specific gene $i$ and cell $j$ ($x_{ij}^{obs}$) to calculate the probability distribution of the true number of RNA counts for that gene and cell ($x_{ij}^{true}$). Using a Bayesian framework will allow us to incorporate prior estimates of read counts to help fill in (impute) missing values in a principled manner. Then we can determine the most probable value for $x_{ij}^{true}$ and use that normalized data for downstream analysis.

In this problem, you'll walk through the steps of the Bayesian inference process, often zooming in on a particular cell or gene (since applying these procedures across the entire dataset would take a fair bit of compute).

We'll use a [scRNA-seq dataset](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4441768) generated on a population of mouse embryonic stem cells (mESCs) from the Klein lab.

**Problem 1a (1 point)**

Load the raw scRNA-seq data using the code below.

Each column contains data from a single cell. Each row corresponds to a given gene, and **the gene names are stored as the row index** (the row labels of the DataFrame — you can see them in the leftmost column when you call `.head()`).

How many cells and genes were sequenced in this experiment?

*Hint: `df.shape` returns `(num_rows, num_columns)`*

In [ ]:
raw_data = pd.read_csv('Q1_mESC_raw_data.csv.gz', index_col=0, header=None)
raw_data.head()

In [ ]:
# TODO: print the number of genes and cells
...

**Short Answer 1a** <br>
<font color="red">
    TODO
</font>

For this situation, the Bayes equation looks like:

$P(x_{ij}^{true} | x_{ij}^{obs}) = \frac{P(x_{ij}^{obs} | x_{ij}^{true}) \cdot P(x_{ij}^{true})}{P(x_{ij}^{obs})}$

Let's break down how the authors modeled each of these probability distributions.

**Problem 1b: Likelihood (2 points)**

The likelihood term is $P(x_{ij}^{obs} | x_{ij}^{true})$. It describes the distribution of observed RNA counts given the true amount of RNA in a single cell.

Each RNA molecule in the cell is independently either captured (with probability $\beta_j$, a cell-specific capture efficiency) or missed. Given $x_{ij}^{true}$ total RNA molecules for gene $i$ in cell $j$:

**(i)** What probability distribution describes the number of observed (captured) RNAs? *(If you're unsure, see the paper linked above.)*

**(ii)** Write the equation for the likelihood $P(x_{ij}^{obs} | x_{ij}^{true})$ in terms of $\beta_j$, $x_{ij}^{true}$, and $x_{ij}^{obs}$.

**Short Answer 1b.i (1 point)** <br>
<font color="red">
    TODO
</font>

**Short Answer 1b.ii (1 point)** <br>
<font color="red">
    TODO
</font>

**Problem 1c: Prior**

The prior term is $P(x_{ij}^{true})$, and it describes our prior belief about what the true RNA count for gene $i$ would be in a given cell.

We'll pool the expression data across all cells in the dataset to parameterize a prior distribution for each gene. This prior will help fill in missing values.

To capture the effects of bursty transcription, the authors chose to model the true number of RNAs in a cell using a [negative binomial distribution](https://en.wikipedia.org/wiki/Negative_binomial_distribution). This distribution has two parameters:
- **$r$** (sometimes called the "number of successes"): controls the shape of the distribution. In the context of transcriptional bursting, it relates to the number of transcriptional bursts.
- **$p$** (the "success probability"): controls the mean-to-variance ratio. Values of $p$ close to 1 produce distributions tightly concentrated around zero, while smaller values of $p$ produce distributions with larger means and heavier tails.

The values for these parameters depend on the expression level of a given gene, so we'd need to calculate $p_i$ and $r_i$ for every gene. (To generate the final prior, the gene-specific capture efficiency also needs to be taken into account, but here we'll focus on the calculation of $p$ and $r$.)

In this problem, we'll perform this process for a single gene: **Trim28**.

*Recall from the pandas guide that you can access a row by its label: `raw_data.loc['Trim28']` returns the expression values for Trim28 across all cells.*

To estimate $p$ and $r$ for Trim28, we'll use a process known as the **Method of Moments** — this means we'll use our sample mean and variance, along with the theoretical equations for the mean and variance of a negative binomial, to solve for $p$ and $r$.

**1c.i (1 point)**

Calculate and print the mean and variance of Trim28 expression across all cells in the mESC dataset.

*Hint: pandas Series have built-in `.mean()` and `.var()` methods, e.g. `raw_data.loc['Trim28'].mean()`*

In [ ]:
# TODO: calculate and print the mean and variance of Trim28 expression
...

**1c.ii (2 points)**

The theoretical mean and variance of a negative binomial are:

$\text{mean} = \frac{r(1-p)}{p}$

$\text{variance} = \frac{r(1-p)}{p^2}$

By dividing the mean by the variance, we can solve for $p$ and $r$ analytically:

$$p = \frac{\text{mean}}{\text{variance}}, \quad r = \frac{p \cdot \text{mean}}{1 - p}$$

Use these formulas to compute $p$ and $r$ for Trim28 from the mean and variance you calculated above.

In [ ]:
# TODO: compute p and r using the formulas above
p_fit = ...
r_fit = ...
print(f"r = {r_fit:.1f}, p = {p_fit:.1f}")

**1c.iii (2 points)**

To check that the fit looks good, plot a histogram of Trim28 expression across all cells and overlay the PDF for a negative binomial with the parameters you solved for above.

- Plot the histogram using `plt.hist(..., density=True)` or `sns.histplot(..., stat='density')`
- Use `st.nbinom(n=r_fit, p=p_fit).pmf(x_vals)` to compute the negative binomial PDF at a range of x values
- Use `plt.plot(x_vals, y_vals)` to overlay the PDF

*Hint: plot the histogram first to get a sense of a reasonable x-range, then use `np.arange()` to generate x values.*

In [ ]:
plt.figure(figsize=(6, 5))

# Plot negative binomial fit
x_vals = np.arange(0, 100, 1)
y_vals = ...  # TODO
plt.plot(x_vals, y_vals, label='Negative binomial fit')

# Plot histogram of Trim28 expression
sns.histplot(..., stat='density', label='Trim28 expression')  # TODO

plt.xlabel('Trim28 expression')
plt.ylabel('Density')
plt.legend()

**Problem 1d: The posterior distribution**

The posterior is $P(x_{ij}^{true} | x_{ij}^{obs})$. It combines the likelihood from 1b (how likely are our observed counts given a true count?) with the prior from 1c (what do we expect the true count to be based on other cells?) to produce a probability distribution over possible true RNA counts.

In other words, **bayNorm uses the binomial likelihood you identified in 1b together with the negative binomial prior from 1c, and applies Bayes' theorem to compute a posterior distribution for each gene–cell pair.** The result tells us which values of $x_{ij}^{true}$ are most probable given our data and prior knowledge.

To collapse the posterior distribution to a single "best guess" value, we use the **MAP (Maximum a Posteriori) estimate** — the value of $x_{ij}^{true}$ that has the highest posterior probability.

For a theoretical PDF, the MAP is the value that maximizes the PDF. But Bayesian analysis often relies on simulated samples rather than closed-form equations. In that case, we can approximate the MAP as the center of the most populated bin in a histogram of samples.

**1d.i (2 points)**

We've provided a set of simulated random samples from the posterior distribution for Trim28 in cell 0. Plot a histogram of the data with the given bins. What is the MAP for $x_{Trim28, cell\_0}^{true}$?

*Hint: identify the tallest bin and report its center value.*

In [ ]:
trim28_posterior_samples = [370,345,298,335,276,265,321,206,291,317,
                  442,381,526,204,315,294,356,460,403,264,
                  176,272,184,196,305,277,249,243,402,397,
                  374,417,252,385,443,180,373,216,304,191,
                  357,360,282,318,268,351,305,308,181,346]

In [ ]:
# Creates 6 evenly spaced bin edges between the min and max
hist_bins = np.linspace(np.min(trim28_posterior_samples),
                        np.max(trim28_posterior_samples), 6)

plt.figure(figsize=(6, 5))
# TODO: plot a histogram of trim28_posterior_samples using hist_bins
...

plt.xlabel('Trim28 posterior samples')
plt.ylabel('Density')

**Short Answer 1d.i** <br>
<font color="red">
    TODO
</font>

**1d.ii (2 points)**

We ran the bayNorm package on this dataset to generate the MAP estimates for all gene–cell pairs. Load this data using the cell below.

To visualize how the MAPs compare to the raw sequencing counts, **choose one cell** and make a scatter plot of the MAP estimates ($x_{ij}^{true}$, y-axis) vs. the raw counts ($x_{ij}^{obs}$, x-axis) across all genes for that cell.

*Hint: to plot data for cell 1, use `raw_data[1]` for raw counts and `maps[1]` for MAP estimates.*

Why would the MAPs be higher than the raw scRNA-seq values? Give a rough estimate of the capture efficiency for this experiment.

In [ ]:
maps = pd.read_csv('Q1_MAPs.csv.gz', index_col=0, skiprows=1, names=range(1, 934))

# Verify it's the same shape as raw_data
print(f"raw_data shape: {raw_data.shape}, maps shape: {maps.shape}")

In [ ]:
plt.figure(figsize=(7, 6))
# Pick a cell (e.g. cell 1). raw_data[1] gives raw counts, maps[1] gives MAP estimates.
plt.scatter(..., ..., alpha=0.5)  # TODO

plt.xlabel('Raw sequencing counts')
plt.ylabel('MAP estimate')

**Short Answer 1d.ii** <br>
<font color="red">
    TODO
</font>

**1d.iii (1 point)**

Recreate the plot above, but this time log-scale both axes by adding the line `plt.loglog()`. This will make it easier to see the relationship at low sequencing counts.

Does the prior have a stronger impact on the MAP estimate for genes with high or low raw sequencing counts? How can you tell?

In [ ]:
# TODO: recreate the scatter plot with log-scaled axes
...

**Short Answer 1d.iii** <br>
<font color="red">
    TODO
</font>

# Problem 2: Dimensionality reduction of scRNA-seq data (13 points)

In this problem, we'll explore how to use PCA to reduce the dimensionality of single-cell RNA-seq data and identify clusters.

**Background (if you skipped Problem 1):** The dataset comes from a single-cell RNA-seq experiment on mouse embryonic stem cells. The raw sequencing counts were normalized using a Bayesian method (bayNorm) that estimates the true RNA counts for each gene in each cell. The result is a matrix called `maps` where each row is a gene and each column is a cell — the values are the MAP (Maximum a Posteriori) estimates of true RNA counts.

*Run the cell below to load this data. If you already loaded `maps` in Problem 1d.ii, you can skip it.*

In [ ]:
# Load MAP-normalized data (run this if you skipped Problem 1)
maps = pd.read_csv('Q1_MAPs.csv.gz', index_col=0, skiprows=1, names=range(1, 934))
print(f"maps shape: {maps.shape}")

**PCA**

To visualize differences in gene expression across all individual cells, we'll reduce the dimensionality using PCA.

In part (a), you'll walk through the steps to apply PCA in Python.

**Step 1: Standardize data**

The first step is to standardize the data so that each gene has mean 0 and standard deviation 1 across all cells. This prevents highly expressed genes from dominating the PCA.

The code below uses `StandardScaler` from sklearn to standardize the MAPs dataset. Note that we transpose (`maps.T`) because sklearn expects samples (cells) as rows and features (genes) as columns.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
standardized_data = scaler.fit_transform(maps.T)

**Step 2: Perform PCA on standardized data**

Run the code below to perform PCA.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA()
pca.fit(standardized_data)

**Step 3: Interpret PCA**

PCA produces a set of orthogonal principal components (PCs) that are linear combinations of the original variables (gene expression levels). They are ordered so that PC1 explains the most variance, PC2 the second-most, etc.

**2a (2 points)**

`pca.explained_variance_ratio_` returns an array with the fraction of variance explained by each PC. Plot the **cumulative** variance explained as a function of the number of PCs. *(So at $x=3$, the y-value should be the total variance explained by PC1 + PC2 + PC3.)*

How many principal components are required to explain 80% of the variance?

*Hint: `np.cumsum()` computes a cumulative sum.*

In [ ]:
evar = pca.explained_variance_ratio_
cum_var = ...  # TODO

plt.plot(np.arange(1, len(evar) + 1), cum_var)
plt.xlabel('Number of PCs')
plt.ylabel('Cumulative explained variance')
plt.axhline(y=0.8, color='r', linestyle='--', label='80%')
plt.legend()

**2a Short Answer**

<font color="red">
    TODO
</font>

**2b (2 points)**

PC1 is a set of weights (also called **loadings**) $\alpha_1, \alpha_2, ..., \alpha_n$ — one per gene — such that the weighted sum $\alpha_1 g_1 + \alpha_2 g_2 + ... + \alpha_n g_n$ captures the direction of greatest variance across cells. ($g_n$ is the expression level of gene $n$.)

What does it mean if $\alpha_x$ for gene $x$ is close to 0?

**Short Answer 2b** <br>
<font color="red">
    TODO
</font>

**2c (2 points)**

`pca.components_[0]` returns an array of all the loadings ($\alpha$'s) for PC1. Plot these values and determine which gene contributes the most to PC1.

*Hint: you want the gene with the largest loading **in absolute value**. `np.argmax(np.abs(...))` gives the index. Then use `maps.index[i]` to get the gene name.*

In [ ]:
# TODO: plot PC1 loadings and identify the gene with the largest absolute loading
...

**2c Short Answer**

<font color="red">
    TODO
</font>

**2d (2 points)**

Let's see what these cells look like in the reduced PCA space. Run the code below to project the data onto the principal components and plot PC1 vs PC2.

Then, identify which genes drive the apparent clustering along PC2 (see "2d continued" below).

In [ ]:
# Project data onto principal components
pcaProj = pca.fit_transform(standardized_data)

In [ ]:
sns.jointplot(x=pcaProj[:, 0], y=pcaProj[:, 1], kind='hex')
plt.xlabel('PC1')
plt.ylabel('PC2')

**2d continued**

You should see what looks like two clusters of cells separated along PC2. To figure out which genes drive this separation, we want to find the genes with the highest absolute loadings in PC2.

The code below creates a DataFrame with gene names as the index, a `loading` column with the PC2 loading for each gene, and a `median_exp` column with the median MAP expression across all cells.

**Your task:** Add a column with the absolute value of the loading, then sort and print the 20 genes with the largest absolute PC2 loadings.

*Hint: `df['abs_load'] = np.abs(df['loading'])` adds a new column, and `df.sort_values('abs_load', ascending=False)[:20]` gives the top 20 rows.*

In [ ]:
# DataFrame of PC2 loadings and median expression
pc2_loadings_df = pd.DataFrame(pca.components_[1], index=maps.index, columns=['loading'])
pc2_loadings_df['median_exp'] = maps.median(axis=1)
pc2_loadings_df.head()

In [ ]:
# TODO: find and print the 20 genes with the largest absolute PC2 loadings
...

**Problem 2e (1 point)**

The gene with the highest absolute loading in PC2 should be **Mir6975**. To visualize whether expression of this microRNA correlates with the clusters you see in PCA, run the cell below to color the PC1 vs PC2 scatter plot by Mir6975 expression.

Is Mir6975 expression strongly or weakly correlated with the clusters along PC2?

In [ ]:
# Plot data colored by Mir6975 expression
sns.scatterplot(x=pcaProj[:, 0], y=pcaProj[:, 1],
                hue=maps.loc['Mir6975'], palette='viridis',
                marker='.', edgecolor='none', alpha=0.7)
plt.xlabel('PC1')
plt.ylabel('PC2')

**2e Short Answer**

<font color="red">
    TODO
</font>

**Problem 2f (3 points)**

You're excited about these potential clusters, but you notice that the Mir6975 MAP-adjusted counts are quite low. Is that true for the other genes with high PC2 loadings?

To investigate, make two side-by-side histograms:
1. (Left) Distribution of median MAP expression for the 20 genes from part 2d
2. (Right) Distribution of median MAP expression across **all** genes

Remember that `pc2_loadings_df` has a `median_exp` column with the median MAP counts for each gene.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

# Left: median MAP expression for top 20 PC2 loading genes
top20 = pc2_loadings_df.sort_values(..., ascending=False)[:20]  # TODO
sns.histplot(top20['median_exp'], stat='density', ax=ax1)
ax1.set_xlabel('Median MAP RNA level')
ax1.set_title('Top 20 PC2 loadings')

# Right: median MAP expression across all genes
sns.histplot(pc2_loadings_df['median_exp'], stat='density', ax=ax2)
ax2.set_xlabel('Median MAP RNA level')
ax2.set_title('All genes')
ax2.set_xlim((-10, 150))

plt.tight_layout()

**2f Short Answer**

<font color="red">
    TODO
</font>

**Problem 2g (1 point)**

Does your answer above change your confidence in the clustering along PC2? Why or why not?

*(There's not one "right" answer — share a few thoughts and/or reactions.)*

**2g Short Answer**

<font color="red">
    TODO
</font>

# Problem 3: Differential expression analysis, multiple hypothesis testing, and false discovery rates (13 points)

### Background

In this problem, we will explore [a dataset measuring bacterial mRNA and protein expression across different media conditions](https://www.nature.com/articles/srep45303). To simplify the analysis, we have extracted a subset of the RNA-seq data. In the provided file (`Q3.csv.gz`), each row contains the observed RNA level for a given gene, and each column corresponds to a different experimental sample. Our aim is to identify genes whose expression changes significantly between **glucose** and **glycerol** growth media.

**Problem 3a.i (1 Point): Hierarchical clustering**

Load the dataset and generate a clustered heatmap with experimental conditions along the x-axis and genes along the y-axis, with both axes hierarchically clustered.

*Hint: [`sns.clustermap()`](https://seaborn.pydata.org/generated/seaborn.clustermap.html) will do most of the work for you.*

In [ ]:
df = pd.read_csv('Q3.csv.gz', index_col=0)
df.head()

In [ ]:
# TODO: generate a clustered heatmap
...

**Problem 3a.ii (2 Points)**

Assuming that the samples were collected and processed in pairs (e.g. glu1 and gly1 were co-processed, glu2 and gly2 were co-processed), are the batch effects stronger or weaker than the biological response to a change in carbon source? Explain your reasoning based on the clustering pattern.

**Short Answer**: <br>
<font color="red">
    TODO
</font>

**Problem 3b (2 Points)**

Using a two-sample t-test, compute a p-value for each gene testing the difference in mean expression between glucose and glycerol conditions. What is the smallest p-value you observe?

Use `ttest_ind()` with `equal_var=False` (Welch's t-test). Do not correct for multiple hypothesis testing.

*Hint: You should get one p-value per gene.*

In [ ]:
# Column names for each condition
glu_columns = ['glu1', 'glu2', 'glu6', 'glu7', 'glu11', 'glu12']
gly_columns = ['gly1', 'gly2', 'gly6', 'gly7', 'gly11', 'gly12']

In [ ]:
# df[glu_columns] selects just the glucose columns from the dataframe
result = ttest_ind(df[...], df[...], axis=1, equal_var=False)  # TODO: which columns?
pvals = result.pvalue

print(f'Smallest p-value: {pvals.min():.2e}')

**Problem 3c (2 Points): Volcano plot**

Make a scatter plot of $-\log_{10}$(p-value) vs. $\log_2$(average\_mRNA\_gly / average\_mRNA\_glu) for each gene. This is called a **volcano plot**.

Steps:
1. Calculate the mean expression across glycerol replicates and glucose replicates for each gene
2. Compute $\log_2$(mean\_gly / mean\_glu) — this is the **log2 fold-change**
3. Plot $-\log_{10}$(p-value) on the y-axis vs. log2 fold-change on the x-axis

We are often interested in genes that change more than 2-fold (i.e., $|\log_2\text{FC}| > 1$) with small p-values (e.g., $p < 0.01$, i.e., $-\log_{10}(p) > 2$). Use `plt.axvline()` and `plt.axhline()` to draw lines at these boundaries.

**Interpretation:** Genes in the **upper-left** region of the volcano plot are significantly upregulated in glucose (higher in glucose than glycerol). Genes in the **upper-right** are significantly upregulated in glycerol. What is the biological interpretation of genes found in each of these regions?

In [ ]:
# Compute mean expression per condition
mean_gly = df[gly_columns].mean(axis=1)
mean_glu = df[glu_columns].mean(axis=1)

# Compute log2 fold-change and add to dataframe
df['log2_fc'] = ...  # TODO
df['pvals'] = pvals
df['neg_log10_pval'] = ...  # TODO

# Volcano plot
plt.figure(figsize=(10, 6))
plt.scatter(df['log2_fc'], df['neg_log10_pval'], s=3, alpha=0.5)

# Draw boundary lines for fold-change > 2 and p < 0.01
plt.axhline(...)  # TODO
plt.axvline(...)  # TODO
plt.axvline(...)  # TODO

plt.xlabel('log2(glycerol / glucose)')
plt.ylabel('-log10(p-value)')

**Short Answer**: <br>
<font color="red">
    TODO
</font>

**Problem 3d (2 points): Naive significance testing**

If we used a significance threshold of $\alpha = 0.01$ without correcting for multiple hypothesis testing:

(i) Plot a histogram of $\log_{10}$-transformed p-values and draw a **vertical** line at $\log_{10}(0.01) = -2$ to show the cutoff. Use `binwidth=0.05`.

(ii) For how many genes would we reject the null hypothesis?

(iii) Which of these significant genes change more than 2-fold?

In [ ]:
# Histogram of log10(p-values)
sns.histplot(np.log10(df['pvals']), binwidth=0.05)
plt.axvline(x=..., color='r', linewidth=1)  # TODO: where does alpha = 0.01 fall on a log10 scale?
plt.xlabel('log10(p-value)')

In [ ]:
# (ii) How many genes have p < 0.01?
sig_genes = df[df['pvals'] < ...]  # TODO
print(f'Reject null for {len(sig_genes)} genes')

# (iii) Which of these also change more than 2-fold?
sig_2fold = df[(df['pvals'] < ...) & (np.abs(df['log2_fc']) > ...)]  # TODO
print(f'{len(sig_2fold)} of those have >2-fold change')
print(sig_2fold.index.tolist())

**Problem 3e (2 Points): Bonferroni-corrected significance testing**

Now let's apply a **Bonferroni correction** for multiple hypothesis testing. The Bonferroni-corrected threshold is $\alpha_{\text{Bonf}} = \alpha / n$, where $n$ is the number of tests (genes).

(i) Apply this correction and re-create the histogram from 3d, drawing a vertical line at the corrected $\log_{10}$(p-value) cutoff.

(ii) For how many genes would we reject the null hypothesis, and which of these change more than 2-fold?

In [ ]:
# Bonferroni correction
alpha = 0.01
a_bonf = ...  # TODO

# Plot histogram with corrected cutoff
sns.histplot(np.log10(df['pvals']), binwidth=0.05)
plt.axvline(x=np.log10(a_bonf), color='r', linewidth=1)
plt.xlabel('log10(p-value)')

In [ ]:
# How many genes pass Bonferroni correction?
sig_bonf = df[df['pvals'] < a_bonf]
print(f'Reject null for {len(sig_bonf)} genes')

# Which of these change more than 2-fold?
sig_bonf_2fold = df[(df['pvals'] < a_bonf) & (np.abs(df['log2_fc']) > 1)]
print(f'{len(sig_bonf_2fold)} of those have >2-fold change')

**Problem 3f (2 points): Benjamini-Hochberg-corrected significance testing**

Now let's apply a **Benjamini-Hochberg (BH)** correction, which controls the false discovery rate (FDR).

**(i)** Apply the BH correction and re-create the histogram, drawing a vertical line at the BH-corrected cutoff.

**(ii)** For how many genes would we reject the null hypothesis, and which of these change more than 2-fold?

**(iii)** Print the significant genes with more than 2-fold changes, **sorted by decreasing absolute fold-change**. Look up the top two genes on [EcoCyc](https://ecocyc.org). Do these genes make sense given the experimental conditions (glucose vs. glycerol)? **Explain in 1–2 sentences in the short answer below.**

To implement Benjamini-Hochberg:

1. Sort the p-values in ascending order. Let each p-value $p_j$ have rank $i$ (starting from 1).
2. For each p-value, check whether $p_j < \frac{i}{n} \cdot \alpha$, where $n$ is the total number of tests. If so, it's a "discovery."
3. The largest p-value among all discoveries is your BH-corrected cutoff.

In [ ]:
# Sort p-values in ascending order
pvals_sorted = np.sort(df['pvals'])
n_tests = len(pvals_sorted)
alpha = 0.01

# Find all p-values that pass BH criterion: p_j < (rank / n_tests) * alpha
discoveries = []
for rank, pval in enumerate(pvals_sorted):
    threshold = ...  # TODO
    if pval < threshold:
        discoveries.append(pval)

# BH cutoff is the largest discovered p-value
a_bh = max(discoveries)
print(f'BH cutoff: {a_bh:.2e}')

In [ ]:
# Plot histogram with BH-corrected cutoff (same pattern as 3d and 3e)
sns.histplot(np.log10(df['pvals']), binwidth=0.05)
plt.axvline(x=np.log10(a_bh), color='r', linewidth=1)
plt.xlabel('log10(p-value)')

In [ ]:
# Significant genes with >2-fold change, sorted by absolute fold-change
sig_bh_2fold = df[(df['pvals'] < a_bh) & (np.abs(df['log2_fc']) > 1)]
sig_bh_2fold = sig_bh_2fold.copy()
sig_bh_2fold['abs_fc'] = ...  # TODO
sig_bh_2fold.sort_values('abs_fc', ascending=False)

**Short Answer**: <br>
<font color="red">
    TODO
</font>

# Problem 4: Sequence alignment (13 points)

#### Background

You are inspecting a series of related tRNAs from an uncharacterized genome and wish to generate an alignment of these sequences. Interestingly, you find that these tRNAs are heavily modified. In this problem, we will explore the role of pseudouridine ($\Psi$).

#### 4a (4 Points)

Historically, $\Psi$ is thought to act as a "universal" base — it can pair with A, G, U, or C. As a result, the position of $\Psi$ is highly conserved, and substitutions to other bases are very rarely observed.

Below is a partial **symmetric** substitution scoring matrix (similar to BLOSUM62 but for RNA bases). The `*` row and column represent **gaps** (insertions/deletions): aligning a base against a gap incurs a penalty, while a gap–gap alignment scores +1.

|   | A  | G  | C  | U  | Ψ  | *  |
| - | -  | -  | -  | -  | -  | -  |
| A | +2 | x  | x  | x  | x  | x  |
| G | -2 | +2 | x  | x  | x  | x  |
| C | -2 | -2 | +2 | x  | x  | x  |
| U | -2 | -2 | -2 | +2 | x  | x  |
| Ψ | __ | __ | __ | __ | __ | x  |
| * | -2 | -2 | -2 | -2 | -2 | +1 |

As you can see, matching A with A yields a score of +2, while substituting A with G yields a score of -2, etc.

Based on the information that $\Psi$ is highly conserved and substitutions away from $\Psi$ are very rarely observed, fill in the blanks for the $\Psi$ row with integer values between -3 and +3. Remember that the matrix is symmetric (so the $\Psi$ column follows from the $\Psi$ row).

*Hint: Your scores should reflect that $\Psi \leftrightarrow \Psi$ matches are rewarded, substitutions of $\Psi$ for a standard base are heavily penalized, and $\Psi$ with a gap follows the pattern of other bases.*

**Short Answer**: <br>
<font color="red">
    TODO
</font>

#### 4b (4 Points)

While examining your tRNA sequences, you notice a seemingly conserved region and decide to perform a **local sequence alignment** of this region in two different tRNAs (tRNA-Ala and tRNA-Val) using the **Smith-Waterman algorithm**.

The results of your alignment matrices (both the scoring matrix and the traceback) are shown below, where sequence 1 is along the columns and sequence 2 is along the rows.

**Your task:** Identify the traceback path for local alignment, then provide the aligned sequences with `|` for matches and `-` for gaps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Setup Sequences and Scoring Parameters
S1 = "AGACAΨΨGCACCC"    # Sequence along the top (columns)
S2 = "AGACAΨUUAΨGCA"    # Sequence along the side (rows)
match, mismatch, gap = 2, -2, -2
p_match, p_mismatch = 3, -3  # Scoring for conserved Pseudouridine (Ψ)
FS = 22 # Unified large font size for sequences, numbers, and arrows

# 2. Build Smith-Waterman Matrices (M+1 by N+1)
rows, cols = len(S2) + 1, len(S1) + 1
scores = np.zeros((rows, cols), dtype=int)
dirs = np.zeros((rows, cols), dtype=int) # 1:Diag, 2:Up, 3:Left

for i in range(1, rows):
    for j in range(1, cols):
        c1, c2 = S1[j-1], S2[i-1]
        # Score selection: special handling for Ψ
        s = p_match if (c1 == c2 == 'Ψ') else (p_mismatch if 'Ψ' in (c1, c2) else (match if c1 == c2 else mismatch))

        # Smith-Waterman logic: max of (0, diag, up, left)
        opts = [0, scores[i-1, j-1] + s, scores[i-1, j] + gap, scores[i, j-1] + gap]
        scores[i, j] = max(opts)
        if scores[i, j] > 0:
            dirs[i, j] = np.argmax(opts)

# 3. Visualization logic
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
arrows = {0: "", 1: "↖", 2: "↑", 3: "←"}

# Axis labels (empty string for the initialization row/column)
xticklabels, yticklabels = [""] + list(S1), [""] + list(S2)

for ax, mode in zip([ax1, ax2], ["nums", "dirs"]):
    # Create Annotations (Numbers in Plot 1, Arrows in Plot 2)
    ann = scores.astype(str) if mode == "nums" else np.array([[arrows[v] for v in r] for r in dirs])

    sns.heatmap(scores, annot=ann, fmt="", cmap="YlGnBu", cbar=False, square=True, ax=ax,
                xticklabels=xticklabels, yticklabels=yticklabels, annot_kws={"size": FS},
                linewidths=0.5, linecolor='gray')

    # Styling and Axis layout
    ax.set_title(f"{'Scoring Matrix' if mode=='nums' else 'Traceback Path'}", pad=40, fontsize=FS+4)
    ax.xaxis.tick_top()
    ax.tick_params(left=False, top=False, labelsize=FS)
    plt.setp(ax.get_xticklabels(), rotation=0)
    plt.setp(ax.get_yticklabels(), rotation=0)

    # Complete the external grid borders
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('gray')
        spine.set_linewidth(0.5)

# 4. Highlight the Optimal Alignment Path (Traceback from max score 16)
ci, cj = np.unravel_index(np.argmax(scores), scores.shape)
while ci > 0 and cj > 0 and scores[ci, cj] > 0:
    ax2.add_patch(plt.Rectangle((cj, ci), 1, 1, fill=False, edgecolor='red', lw=4))
    d = dirs[ci, cj]
    if d == 1: ci, cj = ci-1, cj-1
    elif d == 2: ci -= 1
    elif d == 3: cj -= 1
    else: break

plt.tight_layout()
plt.show()

In [ ]:
print("Example format:")
print("A B C D E")
print("| | - | |")
print("A B – D E")

In [ ]:
# TODO: provide your alignment
print("")
print("")
print("")

#### 4c (2 Points)

tRNAs are known to have a series of stem-loops, which contain a variable number of bases in the loop. A colleague suggests that you should alter the alignment protocol so that **opening** a gap has a different (higher) penalty than **extending** a gap. Explain why this change might be well suited to provide better alignments of your tRNAs.

**Short Answer**: <br>
<font color="red"> TODO </font>

#### 4d (3 Points)

(True or False) To implement a two-tiered gap penalty as described in 4c, one simply needs to add a new column and a new row to the substitution matrix (similar to the matrix in part 4a) and no other changes are needed. Explain your answer.

**Short Answer**: <br>
<font color="red"> TODO </font>

---

# Submission Checklist

- [ ] Completed at least 3 of 4 problems (you may skip one)
- [ ] All <font color="red">TODO</font> sections completed for attempted problems
- [ ] All code cells executed (no `...` remaining in attempted problems)
- [ ] Collaborators listed
- [ ] Downloaded as PDF and submitted to Gradescope